In [ ]:
import os
import sys
import re
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# -------- Configuration --------
base_data = "/Users/harshit/Desktop/nature-in-language/data"
country     = "EE" #Change as per country
input_dir  = os.path.join(base_data, "raw", f"ParlaMint-{country}-en.txt")
output_dir = os.path.join(base_data, "results", "metadata", country)
os.makedirs(output_dir, exist_ok=True)

# Validate input directory
if not os.path.isdir(input_dir):
    print(f"Error: The directory {input_dir} does not exist.")
    sys.exit(1)

# -------- Aggregation Initialization --------
gender_data   = []  # list of dicts for each session
align_data    = []
ideology_data = []
generation_data = []

# Define generation ranges (birth years)
gen_ranges = {
    "Silent Generation": (1928, 1945),
    "Baby Boomers":      (1946, 1964),
    "Generation X":      (1965, 1980),
    "Millennials":       (1981, 1996),
    "Generation Z":      (1997, 2012)
}

In [ ]:
# -------- Data Collection --------
# Find all *-meta.tsv files. Use os.walk for robust traversal.
meta_files = []
print("▶︎ [Checkpoint] Starting file discovery…")
for root, _, files in os.walk(input_dir):
    for fname in files:
        if fname.endswith("-meta.tsv"):
            meta_files.append(os.path.join(root, fname))

print(f"▶︎ [Checkpoint] Found {len(meta_files)} metadata files.")

if not meta_files:
    print("No '*-meta.tsv' files found in the given directory.")
    sys.exit(1)

# Simplified session list
sessions = []
print("▶︎ [Checkpoint] Building session list…")
for fpath in meta_files:
    fname = os.path.basename(fpath)
    date_match = re.search(r"\d{4}-\d{2}-\d{2}", fname)
    session_date = date_match.group(0) if date_match else ""
    sessions.append((session_date, fname, fpath))
sessions.sort(key=lambda x: (x[0], x[1]))
print(f"▶︎ [Checkpoint] {len(sessions)} sessions sorted chronologically. First 3:\n  {sessions[:3]}")

In [ ]:
# -------- Processing Each Session --------
for session_date, session_key, fpath in sessions:
    # Read TSV file into DataFrame
    try:
        df = pd.read_csv(fpath, sep='\t', index_col=False)
    except Exception as e:
        print(f"Warning: Could not read {fpath} ({e}), skipping this file.")
        continue

    # Identify speaker identifier column (some files use Speaker_ID)
    if 'Speaker_ID' in df.columns:
        speaker_id_col = 'Speaker_ID'
    elif 'SpeakerId' in df.columns:
        speaker_id_col = 'SpeakerId'
    else:
        # Fallback: if no explicit ID, use Speaker_name
        speaker_id_col = 'Speaker_name' if 'Speaker_name' in df.columns else None

    # Deduplicate speakers within the session
    if speaker_id_col:
        unique_df = df.drop_duplicates(subset=speaker_id_col).copy()
    else:
        unique_df = df.copy()
    total_speakers = len(unique_df)

        # Fill missing values and cast to string so we can safely use .str methods
    for col in ['Speaker_gender', 'Party_status', 'Party_orientation', 'Speaker_birth']:
        if col in unique_df.columns:
            unique_df[col] = unique_df[col].fillna('').astype(str)

    # Count by Gender
    male_count   = int((unique_df['Speaker_gender'].str.upper() == 'M').sum()) if 'Speaker_gender' in unique_df.columns else 0
    female_count = int((unique_df['Speaker_gender'].str.upper() == 'F').sum()) if 'Speaker_gender' in unique_df.columns else 0
    gender_data.append({
        "Session": session_key,
        "Date": session_date,
        "Male": male_count,
        "Female": female_count
    })

    # Count by Political Alignment (Coalition vs Opposition)
    coalition_count = int((unique_df['Party_status'] == 'Coalition').sum()) if 'Party_status' in unique_df.columns else 0
    opposition_count = int((unique_df['Party_status'] == 'Opposition').sum()) if 'Party_status' in unique_df.columns else 0
    align_data.append({
        "Session": session_key,
        "Date": session_date,
        "Coalition": coalition_count,
        "Opposition": opposition_count
    })

    # Count by Political Ideology (Left, Centre, Right)
    left_count = centre_count = right_count = 0
    if 'Party_orientation' in unique_df.columns:
        for orient in unique_df['Party_orientation'].astype(str):
            orientation = orient.lower()
            if 'left' in orientation:
                left_count += 1
            elif 'right' in orientation:
                right_count += 1
            elif 'centre' in orientation or 'center' in orientation:
                centre_count += 1
    ideology_data.append({
        "Session": session_key,
        "Date": session_date,
        "Left": left_count,
        "Centre": centre_count,
        "Right": right_count
    })

    # Count by Generational Cohort
    gen_counts = { "Silent Generation": 0, "Baby Boomers": 0, "Generation X": 0, "Millennials": 0, "Generation Z": 0 }
    if 'Speaker_birth' in unique_df.columns:
        for birth in unique_df['Speaker_birth']:
            if birth in [None, '', '-', 'nan']:  # skip missing or placeholder
                continue
            try:
                year = int(float(birth))  # some birth years might be float or string
            except:
                continue
            for gen_label, (start_year, end_year) in gen_ranges.items():
                if start_year <= year <= end_year:
                    gen_counts[gen_label] += 1
                    break  # found the generation, break out of loop
    generation_entry = {"Session": session_key, "Date": session_date}
    generation_entry.update(gen_counts)
    generation_data.append(generation_entry)

In [ ]:
print(f"\n▶︎ [Checkpoint] Finished processing all sessions.")
print(f"  • Total sessions processed: {len(sessions)}")
print(f"  • Gender entries:   {len(gender_data)}")
print(f"  • Alignment entries: {len(align_data)}")
print(f"  • Ideology entries:  {len(ideology_data)}")
print(f"  • Generation entries: {len(generation_data)}")

In [ ]:
# -------- Save Aggregated Data to CSV --------
# Convert to DataFrames for sorting and output
gender_df    = pd.DataFrame(gender_data)
alignment_df = pd.DataFrame(align_data)
ideology_df  = pd.DataFrame(ideology_data)
generation_df= pd.DataFrame(generation_data)

# Sort each DataFrame by date (and session as secondary) for logical order
for df in [gender_df, alignment_df, ideology_df, generation_df]:
    if 'Date' in df.columns:
        # Convert Date column to datetime for accurate sorting (if not empty)
        if df['Date'].dtype == object or str(df['Date'].dtype).startswith('str'):
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df.sort_values(['Date','Session'], inplace=True)

# Define output file paths
gender_csv    = os.path.join(output_dir, f"{country}_gender_counts.csv")
alignment_csv = os.path.join(output_dir, f"{country}_alignment_counts.csv")
ideology_csv  = os.path.join(output_dir, f"{country}_ideology_counts.csv")
generation_csv= os.path.join(output_dir, f"{country}_generation_counts.csv")

# Save CSV files (excluding the Pandas index)
print(f"▶︎ [Checkpoint] Writing CSVs to {output_dir} …")
gender_df.to_csv(gender_csv, index=False)
alignment_df.to_csv(alignment_csv, index=False)
ideology_df.to_csv(ideology_csv, index=False)
generation_df.to_csv(generation_csv, index=False)
print("  • CSV files written.")

In [ ]:
# -------- Generate Scatter + LOESS Charts --------
plt.style.use('seaborn-v0_8-whitegrid')  # clean style

# Make sure Date is datetime
for df in (gender_df, alignment_df, ideology_df, generation_df):
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])

def plot_scatter_loess(df, value_cols, title, ylabel, outfile, frac=0.3):
    """
    df          : DataFrame, must have 'Date' and each col in value_cols
    value_cols  : list of column names to plot
    frac        : fraction of data used for each local regression window
    """
    plt.figure(figsize=(10, 6), dpi=300)
    
    # Convert dates to numeric for LOWESS
    x_numeric = df['Date'].map(pd.Timestamp.toordinal).values
    
    for col in value_cols:
        if col not in df.columns: 
            continue
        
        y = df[col].values
        
        # Scatter
        plt.scatter(df['Date'], y, s=20, alpha=0.6, label=f"{col} (raw)")
        
        # LOESS smooth
        loess_result = sm.nonparametric.lowess(endog=y, exog=x_numeric, frac=frac, return_sorted=True)
        # loess_result is array of (x_numeric, y_smoothed)
        x_smooth = [pd.Timestamp.fromordinal(int(xi)) for xi in loess_result[:,0]]
        y_smooth = loess_result[:,1]
        plt.plot(x_smooth, y_smooth, linewidth=2, label=f"{col} (LOESS)")

    plt.xlabel("Session Date")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(ncol=2, fontsize='small')
    plt.tight_layout()
    plt.savefig(outfile)
    plt.close()

# Generate the four charts
plot_scatter_loess(
    gender_df, ["Male","Female"],
    f"{country}: Male vs Female Speakers (scatter + LOESS)",
    "Unique Speakers",
    os.path.join(output_dir, f"{country}_gender_scatter_loess.png"),
    frac=0.2
)

plot_scatter_loess(
    alignment_df, ["Coalition","Opposition"],
    f"{country}: Coalition vs Opposition (scatter + LOESS)",
    "Unique Speakers",
    os.path.join(output_dir, f"{country}_alignment_scatter_loess.png"),
    frac=0.2
)

plot_scatter_loess(
    ideology_df, ["Left","Centre","Right"],
    f"{country}: Political Ideology (scatter + LOESS)",
    "Unique Speakers",
    os.path.join(output_dir, f"{country}_ideology_scatter_loess.png"),
    frac=0.2
)

plot_scatter_loess(
    generation_df,
    ["Silent Generation","Baby Boomers","Generation X","Millennials","Generation Z"],
    f"{country}: Generational Cohorts (scatter + LOESS)",
    "Unique Speakers",
    os.path.join(output_dir, f"{country}_generation_scatter_loess.png"),
    frac=0.2
)

print(f"Scatter + LOESS charts saved to {output_dir}")